In [3]:
# imports
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# LOAD DELIVERABLES (ALREADY LAGGED)
df = pd.read_parquet("data/fx_system_df_lagged.parquet")
returns_df = pd.read_parquet("data/fx_returns_df_lagged.parquet")

# CONFIG (FROZEN SPEC)
H = 21
USE_ZSCORE = True
MIN_CROSS_SECTION = 5 

ROLL_WINDOW = 252

# FX MAP
FX_TICKERS = {
    "AUD": {"ticker": "AUDUSD=X", "invert": False},
    "CAD": {"ticker": "CADUSD=X", "invert": True}, 
    "CHF": {"ticker": "CHFUSD=X", "invert": True},
    "EUR": {"ticker": "EURUSD=X", "invert": False},
    "GBP": {"ticker": "GBPUSD=X", "invert": False},
    "JPY": {"ticker": "JPYUSD=X", "invert": True}, 
    "NZD": {"ticker": "NZDUSD=X", "invert": False}
}

# BUILD CARRY
def build_carry_raw(df, fx_map):
    carry = pd.DataFrame(index=df.index)

    for ccy in fx_map.keys():
        carry[ccy] = df[f"{ccy}_rate"] - df["USD_rate"]

    return carry

carry_raw = build_carry_raw(df, FX_TICKERS)

# ALIGN
common_idx = carry_raw.index.intersection(returns_df.index)
carry_signal = carry_raw.loc[common_idx].copy()
returns_df = returns_df.loc[common_idx].copy()

# drop rows with insufficient coverage
valid_counts = carry_signal.notna().sum(axis=1)
carry_signal = carry_signal[valid_counts >= MIN_CROSS_SECTION]
returns_df = returns_df.loc[carry_signal.index]

# NORMALIZE
def normalize_signal(signal):
    if USE_ZSCORE:
        std = signal.std(axis=1)
        std = std.replace(0, np.nan)
        signal = signal.sub(signal.mean(axis=1), axis=0).div(std, axis=0)
    return signal

carry_signal = normalize_signal(carry_signal)

# FORWARD RETURNS
def compute_forward_returns(returns_df, H):
    return returns_df.rolling(H).sum().shift(-H)

future_returns = compute_forward_returns(returns_df, H)

# align again after forward returns
common_idx = carry_signal.index.intersection(future_returns.index)
carry_signal = carry_signal.loc[common_idx]
future_returns = future_returns.loc[common_idx]
returns_df = returns_df.loc[common_idx]

# IC SERIES (STRICT VALIDITY)
def compute_ic_series(signal, future_returns):
    ic_vals = []

    for date in signal.index:
        s = signal.loc[date]
        r = future_returns.loc[date]

        valid = s.notna() & r.notna()

        if valid.sum() < MIN_CROSS_SECTION:
            ic_vals.append(np.nan)
            continue

        if s[valid].std() == 0:
            ic_vals.append(np.nan)
            continue

        ic_vals.append(spearmanr(s[valid], r[valid]).statistic)

    return pd.Series(ic_vals, index=signal.index)

ic_series = compute_ic_series(carry_signal, future_returns)

# DROP NaNs BEFORE ANY TIME SERIES METRICS
ic_series_clean = ic_series.dropna()

# EXPANDING IC
expanding_ic_mean = ic_series_clean.expanding().mean()
expanding_ic_std = ic_series_clean.expanding().std()
expanding_ic_ir = expanding_ic_mean / expanding_ic_std

# ROLLING IC
rolling_ic_mean = ic_series_clean.rolling(ROLL_WINDOW).mean()
rolling_ic_std = ic_series_clean.rolling(ROLL_WINDOW).std()
rolling_ic_ir = rolling_ic_mean / rolling_ic_std

# IC DISTRIBUTION
ic_stats = {
    "IC Mean": ic_series_clean.mean(),
    "IC Median": ic_series_clean.median(),
    "IC > 0 %": (ic_series_clean > 0).mean(),
    "IC IR": ic_series_clean.mean() / ic_series_clean.std()
}

# EFFECTIVE SAMPLE SIZE
n_eff = len(ic_series_clean) / H

# PORTFOLIO
def build_positions(signal):
    rank = signal.rank(axis=1, pct=True)

    long = (rank >= 0.67).astype(float)
    short = (rank <= 0.33).astype(float)

    pos = long - short
    pos = pos.sub(pos.mean(axis=1), axis=0)

    return pos

positions = build_positions(carry_signal)

pnl = (positions.shift(1) * returns_df).sum(axis=1)

sharpe = pnl.mean() / pnl.std() * np.sqrt(252)

# FINAL OUTPUT
results = {
    **ic_stats,
    **rolling_stats,
    "Expanding IC IR (last)": expanding_ic_ir.iloc[-1] if len(expanding_ic_ir) > 0 else np.nan,
    "Rolling IC IR (last)": rolling_ic_ir.iloc[-1] if len(rolling_ic_ir) > 0 else np.nan,
    "Sharpe": sharpe,
    "Obs": len(ic_series_clean),
    "Effective Obs": n_eff
}

results_df = pd.DataFrame([results])

# SAVE
carry_signal.to_parquet("data/carry_signal.parquet")
ic_series.to_frame("IC").to_parquet("data/carry_ic_series.parquet")
pnl.to_frame("pnl").to_parquet("data/carry_pnl.parquet")
results_df.to_parquet("data/carry_summary.parquet")

In [2]:
results_df

,IC Mean,IC Median,IC > 0 %,IC IR,Rolling IC IR Mean,Rolling IC IR Median,Rolling IC IR > 0 %,Expanding IC IR (last),Rolling IC IR (last),Sharpe,Obs,Effective Obs
0,0.049783,0.071429,0.51747,0.098967,0.105309,0.060268,0.584033,0.098967,-0.03199,0.35482,5123,243.952381
